# Error Analysis — Module 2

This notebook standardizes multiple evaluation datasets and runs inference
to produce targeted error analysis with summary charts and qualitative cases.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession

project_path = Path.cwd()
if not (project_path / "conf").exists():
    project_path = project_path.parent
bootstrap_project(project_path)


In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)
for name in ["kedro", "taxomind", "sentence_transformers", "transformers", "urllib3"]:
    logging.getLogger(name).setLevel(logging.ERROR)


## Load standardized targets


In [ ]:
with KedroSession.create(project_path=project_path) as session:
    run_result = session.run(pipeline_name="error_analysis")

    def _unwrap(value):
        return value.load() if hasattr(value, "load") else value

    classifai_targets = _unwrap(run_result.get("error_analysis_classifai_targets"))
    taxonomy_training_targets = _unwrap(run_result.get("error_analysis_taxonomy_training_targets"))
    training_sentences_targets = _unwrap(run_result.get("error_analysis_training_sentences_targets"))


    context = session.load_context()
    tax_df= context.catalog.load("taxonomy_index")

datasets = {
    "classifai_validation_data": classifai_targets,
    #"taxonomy_training": taxonomy_training_targets,
    #"training_sentences": training_sentences_targets,
}


In [ ]:
# Small-sample debug run
SAMPLE_PER_TAXONOMY = 2000
datasets_small = {}

for name, df in datasets.items():
    if "taxonomy_key" not in df.columns:
        datasets_small[name] = df
        continue
    parts = []
    for tax in df["taxonomy_key"].dropna().unique():
        sub = df[df["taxonomy_key"] == tax]
        if len(sub) > SAMPLE_PER_TAXONOMY:
            sub = sub.sample(n=SAMPLE_PER_TAXONOMY, random_state=7)
        parts.append(sub)
    datasets_small[name] = pd.concat(parts, ignore_index=True)

datasets = datasets_small


## Helpers


In [ ]:
def run_inference(taxonomy_key: str, targets_df: pd.DataFrame) -> pd.DataFrame:
    queries = targets_df["query_text"].tolist()
    with KedroSession.create(
        project_path=project_path,
        runtime_params={
            "taxonomy_key": taxonomy_key,
            "inference_query_input": queries,
        },
    ) as session:
        run_result = session.run(pipeline_name="inference")

    def _unwrap(value):
        return value.load() if hasattr(value, "load") else value

    preds = _unwrap(run_result.get("inference_predictions_df"))
    routing_df = _unwrap(run_result.get("inference_routing_df"))
    if routing_df is not None and "routing_result" in routing_df.columns:
        preds = preds.merge(
            routing_df[["query_id", "routing_result"]],
            on="query_id",
            how="left",
        )
    return preds.copy()


In [ ]:
def load_taxonomy_maps(keys):
    with KedroSession.create(project_path=project_path) as session:
        context = session.load_context()
        partitions = context.catalog.load("taxonomy_index")
    maps = {}
    for key in keys:
        if key not in partitions:
            continue
        df = partitions[key]()
        parent_map = {}
        level_map = {}
        label_map = {}
        for _, row in df.iterrows():
            parent = row["parentCode"]
            if pd.isna(parent) or str(parent).strip() == "":
                parent = "__root__"
            code = str(row["code"]).strip()
            label = row.get("label")
            if pd.notna(label):
                label_map[code] = str(label)
            parent_map[code] = str(parent).strip()
            try:
                level_map[code] = int(row["level"])
            except (TypeError, ValueError):
                continue
        maps[key] = {"parent_map": parent_map, "level_map": level_map, "label_map": label_map}
    return maps


def is_ancestor(ancestor: str, code: str, parent_map: dict) -> bool:
    if not ancestor or not code:
        return False
    if ancestor == "__root__":
        return True
    current = code
    while current and current != "__root__":
        if current == ancestor:
            return True
        current = parent_map.get(current)
    return False


def find_level_ancestor(code: str, target_level: int, parent_map: dict, level_map: dict) -> str:
    current = code
    while current and current != "__root__":
        if level_map.get(current) == target_level:
            return current
        current = parent_map.get(current)
    return ""


def evaluate_predictions(preds, targets, parent_map, level_map, dataset_name, taxonomy_key):
    merged = preds.merge(
        targets[["query_id", "target_code", "target_level", "dataset", "taxonomy_key"]],
        on="query_id",
        how="left",
    )
    valid = merged[merged["target_code"].fillna("").str.strip() != ""].copy()
    valid = valid[valid["predicted_code"].fillna("").str.strip() != ""].copy()
    if "query_text" not in valid.columns:
        if "query" in valid.columns:
            valid["query_text"] = valid["query"]
        else:
            valid["query_text"] = ""
    if "taxonomy_key" not in valid.columns:
        if "taxonomy_key_y" in valid.columns:
            valid["taxonomy_key"] = valid["taxonomy_key_y"]
        elif "taxonomy_key_x" in valid.columns:
            valid["taxonomy_key"] = valid["taxonomy_key_x"]
    valid["exact_match"] = valid["predicted_code"] == valid["target_code"]
    valid["level_match"] = valid["predicted_level"] == valid["target_level"]
    for level in (1, 2, 3, 4):
        target_col = f"target_level_{level}"
        pred_col = f"predicted_level_{level}"
        valid[target_col] = valid["target_code"].map(
            lambda c, lvl=level: find_level_ancestor(str(c).strip(), lvl, parent_map, level_map)
        )
        valid[pred_col] = valid["predicted_code"].map(
            lambda c, lvl=level: find_level_ancestor(str(c).strip(), lvl, parent_map, level_map)
        )
        valid[f"level{level}_match"] = (valid[pred_col] != "") & (valid[pred_col] == valid[target_col])
    valid["under_spec"] = valid.apply(
        lambda r: is_ancestor(r["predicted_code"], r["target_code"], parent_map)
        and r["predicted_code"] != r["target_code"],
        axis=1,
    )
    valid["over_spec"] = valid.apply(
        lambda r: is_ancestor(r["target_code"], r["predicted_code"], parent_map)
        and r["predicted_code"] != r["target_code"],
        axis=1,
    )
    valid["ancestor_match"] = valid.apply(
        lambda r: is_ancestor(r["predicted_code"], r["target_code"], parent_map)
        or r["predicted_code"] == r["target_code"],
        axis=1,
    )
    valid["wrong_branch"] = ~(valid["under_spec"] | valid["over_spec"] | valid["exact_match"])
    valid["right_branch"] = valid["under_spec"] | valid["over_spec"] | valid["exact_match"]

    summary = {
        "dataset": dataset_name,
        "taxonomy": taxonomy_key,
        "rows_evaluated": len(valid),
        "exact_match_rate": valid["exact_match"].mean() if len(valid) else 0.0,
        "level_match_rate": valid["level_match"].mean() if len(valid) else 0.0,
        "level1_match_rate": valid["level1_match"].mean() if len(valid) else 0.0,
        "level2_match_rate": valid["level2_match"].mean() if len(valid) else 0.0,
        "level3_match_rate": valid["level3_match"].mean() if len(valid) else 0.0,
        "level4_match_rate": valid["level4_match"].mean() if len(valid) else 0.0,
        "ancestor_match_rate": valid["ancestor_match"].mean() if len(valid) else 0.0,
        "over_spec_rate": valid["over_spec"].mean() if len(valid) else 0.0,
        "under_spec_rate": valid["under_spec"].mean() if len(valid) else 0.0,
        "right_branch_rate": valid["right_branch"].mean() if len(valid) else 0.0,
        "wrong_branch_rate": valid["wrong_branch"].mean() if len(valid) else 0.0,
        "ambiguous_rate": valid["ambiguous"].mean() if len(valid) else 0.0,
    }

    level_summary = (
        valid.groupby("target_level")
        .agg(
            rows=("query_id", "size"),
            exact_match_rate=("exact_match", "mean"),
            level1_match_rate=("level1_match", "mean"),
            level2_match_rate=("level2_match", "mean"),
            level3_match_rate=("level3_match", "mean"),
            level4_match_rate=("level4_match", "mean"),
            ancestor_match_rate=("ancestor_match", "mean"),
            under_spec_rate=("under_spec", "mean"),
            over_spec_rate=("over_spec", "mean"),
            right_branch_rate=("right_branch", "mean"),
            ambiguous_rate=("ambiguous", "mean"),
        )
        .reset_index()
    )
    level_summary["dataset"] = dataset_name
    level_summary["taxonomy"] = taxonomy_key
    return valid, summary, level_summary


def build_path(code: str, parent_map: dict) -> list:
    code = str(code).strip()
    if not code:
        return ["__root__"]
    path = []
    current = code
    seen = set()
    while current and current not in seen:
        path.append(current)
        seen.add(current)
        if current == "__root__":
            break
        current = parent_map.get(current)
    if not path or path[-1] != "__root__":
        path.append("__root__")
    return list(reversed(path))


def first_divergence(pred_path: list, target_path: list):
    if not pred_path or not target_path:
        return None, "__root__", "", ""
    min_len = min(len(pred_path), len(target_path))
    idx = None
    for i in range(min_len):
        if pred_path[i] != target_path[i]:
            idx = i
            break
    if idx is None:
        if len(pred_path) == len(target_path):
            return None, pred_path[-1], "", ""
        idx = min_len
    parent = pred_path[idx - 1] if idx > 0 else "__root__"
    pred_child = pred_path[idx] if idx < len(pred_path) else ""
    target_child = target_path[idx] if idx < len(target_path) else ""
    return idx, parent, pred_child, target_child


def top_siblings_from_trace(routing_result, parent_code: str, top_n: int = 3):
    if not isinstance(routing_result, dict):
        return []
    trace = routing_result.get("routing_trace") or []
    for entry in trace:
        if entry.get("parent") == parent_code:
            scores = entry.get("scores") or {}
            return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return []


def build_wrong_branch_report(eval_df: pd.DataFrame, taxonomy_maps: dict) -> pd.DataFrame:
    rows = []
    for _, row in eval_df.iterrows():
        if not row.get("wrong_branch"):
            continue
        taxonomy = row.get("taxonomy")
        maps = taxonomy_maps.get(taxonomy, {})
        parent_map = maps.get("parent_map", {})
        level_map = maps.get("level_map", {})
        label_map = maps.get("label_map", {})
        pred_code = str(row.get("predicted_code", "")).strip()
        target_code = str(row.get("target_code", "")).strip()
        if not pred_code or not target_code:
            continue

        pred_path = build_path(pred_code, parent_map)
        target_path = build_path(target_code, parent_map)
        div_idx, div_parent, pred_child, target_child = first_divergence(pred_path, target_path)

        divergence_level = None
        if pred_child and pred_child in level_map:
            divergence_level = level_map[pred_child]
        elif target_child and target_child in level_map:
            divergence_level = level_map[target_child]
        elif div_idx is not None:
            divergence_level = div_idx

        if div_parent == "__root__":
            parent_level = 0
        else:
            parent_level = level_map.get(div_parent)

        top_siblings = top_siblings_from_trace(row.get("routing_result"), div_parent, top_n=3)
        formatted_siblings = []
        for code, score in top_siblings:
            label = label_map.get(str(code).strip(), "")
            if label:
                formatted_siblings.append(f"{code} ({score:.3f}) {label}")
            else:
                formatted_siblings.append(f"{code} ({score:.3f})")

        rows.append({
            "dataset": row.get("dataset", ""),
            "taxonomy": taxonomy,
            "query_text": row.get("query_text", ""),
            "target_code": target_code,
            "predicted_code": pred_code,
            "target_path": " > ".join(target_path),
            "predicted_path": " > ".join(pred_path),
            "divergence_level": divergence_level,
            "divergence_parent": div_parent,
            "divergence_parent_level": parent_level,
            "predicted_child_at_divergence": pred_child,
            "target_child_at_divergence": target_child,
            "top3_siblings": formatted_siblings,
            "stopping_reason": row.get("stopping_reason", ""),
        })

    return pd.DataFrame(rows)



## Run inference and evaluate


In [ ]:
taxonomy_maps = load_taxonomy_maps(["ISCO", "ISIC"])

all_eval = []
all_summaries = []
all_levels = []

all_eval_routing = []
all_summaries_routing = []
all_levels_routing = []

for dataset_name, targets in datasets.items():
    if "taxonomy_key" not in targets.columns:
        print(f"Skipping {dataset_name}: missing taxonomy_key column")
        continue
    for taxonomy_key in sorted(targets["taxonomy_key"].unique()):
        subset = targets[targets["taxonomy_key"] == taxonomy_key].copy()
        subset = subset.reset_index(drop=True)
        subset["query_id"] = subset.index
        preds = run_inference(taxonomy_key, subset)

        eval_df, summary, level_df = evaluate_predictions(
            preds, subset, taxonomy_maps[taxonomy_key]["parent_map"], taxonomy_maps[taxonomy_key]["level_map"], dataset_name, taxonomy_key
        )
        eval_df["dataset"] = dataset_name
        eval_df["taxonomy"] = taxonomy_key
        summary["variant"] = "validated"
        level_df["variant"] = "validated"
        all_eval.append(eval_df)
        all_summaries.append(summary)
        all_levels.append(level_df)

        if "routing_result" in preds.columns:
            routing_only = preds.copy()
            routing_only["predicted_code"] = routing_only["routing_result"].map(
                lambda r: r.get("predicted_code") if isinstance(r, dict) else None
            )
            routing_only["predicted_level"] = routing_only["routing_result"].map(
                lambda r: r.get("predicted_level") if isinstance(r, dict) else None
            )
            routing_only["ambiguous"] = routing_only["routing_result"].map(
                lambda r: r.get("ambiguous") if isinstance(r, dict) else None
            )
            routing_only["stopping_reason"] = routing_only["routing_result"].map(
                lambda r: r.get("stopping_reason") if isinstance(r, dict) else None
            )

            eval_df_r, summary_r, level_df_r = evaluate_predictions(
                routing_only, subset, taxonomy_maps[taxonomy_key]["parent_map"], taxonomy_maps[taxonomy_key]["level_map"], dataset_name, taxonomy_key
            )
            eval_df_r["dataset"] = dataset_name
            eval_df_r["taxonomy"] = taxonomy_key
            summary_r["variant"] = "routing_only"
            level_df_r["variant"] = "routing_only"
            all_eval_routing.append(eval_df_r)
            all_summaries_routing.append(summary_r)
            all_levels_routing.append(level_df_r)

summary_df = pd.DataFrame(all_summaries)
summary_df_routing = pd.DataFrame(all_summaries_routing)
level_df = pd.concat(all_levels, ignore_index=True) if all_levels else pd.DataFrame()
level_df_routing = (
    pd.concat(all_levels_routing, ignore_index=True) if all_levels_routing else pd.DataFrame()
)

summary_df_compare = pd.concat([summary_df, summary_df_routing], ignore_index=True)
level_df_compare = pd.concat([level_df, level_df_routing], ignore_index=True)
summary_df_compare


In [ ]:
# Compare validated vs routing-only
compare_cols = [
    "exact_match_rate",
    "wrong_branch_rate",
    "right_branch_rate",
    "under_spec_rate",
    "over_spec_rate",
    "level1_match_rate",
    "level2_match_rate",
    "level3_match_rate",
    "level4_match_rate",
]

comparison_table = summary_df_compare.pivot_table(
    index=["dataset", "taxonomy"],
    columns="variant",
    values=compare_cols,
)
comparison_table


## Charts


In [ ]:
if not level_df.empty:
    for taxonomy in sorted(level_df["taxonomy"].unique()):
        subset = level_df[level_df["taxonomy"] == taxonomy]
        fig, ax = plt.subplots(figsize=(8, 4))
        for dataset_name in subset["dataset"].unique():
            ds = subset[subset["dataset"] == dataset_name]
            ax.plot(ds["target_level"], ds["exact_match_rate"], marker="o", label=dataset_name)
        ax.set_title(f"Exact match rate by level ({taxonomy})")
        ax.set_xlabel("Target level")
        ax.set_ylabel("Exact match rate")
        ax.set_xticks([1, 2, 3, 4])
        ax.set_ylim(0, 1)
        ax.legend()
        plt.show()

    fig, ax = plt.subplots(figsize=(8, 4))
    summary_plot = summary_df.copy()
    summary_plot["under_spec_rate"] = summary_plot["under_spec_rate"].fillna(0)
    summary_plot["over_spec_rate"] = summary_plot["over_spec_rate"].fillna(0)
    summary_plot["wrong_branch_rate"] = summary_plot["wrong_branch_rate"].fillna(0)
    x = range(len(summary_plot))
    ax.bar(x, summary_plot["under_spec_rate"], label="under_spec")
    ax.bar(x, summary_plot["over_spec_rate"], bottom=summary_plot["under_spec_rate"], label="over_spec")
    bottom = summary_plot["under_spec_rate"] + summary_plot["over_spec_rate"]
    ax.bar(x, summary_plot["wrong_branch_rate"], bottom=bottom, label="wrong_branch")
    ax.set_xticks(list(x))
    ax.set_xticklabels(summary_plot[["dataset", "taxonomy"]].agg(" / ".join, axis=1), rotation=45, ha="right")
    ax.set_ylabel("Rate")
    ax.set_title("Error type rates by dataset")
    ax.set_ylim(0, 1)
    ax.legend()
    plt.tight_layout()
    plt.show()


## Qualitative cases

Pick a few examples for manual inspection.


In [ ]:
if "all_eval" not in globals():
    all_eval = []

all_eval_df = pd.concat(all_eval, ignore_index=True) if all_eval else pd.DataFrame()

def sample_cases(df, taxonomy, dataset, category, n=5):
    subset = df[(df["taxonomy"] == taxonomy) & (df["dataset"] == dataset)]
    if category == "under_spec":
        subset = subset[subset["under_spec"]]
    elif category == "over_spec":
        subset = subset[subset["over_spec"]]
    elif category == "wrong_branch":
        subset = subset[subset["wrong_branch"]]
    else:
        subset = subset[subset["ambiguous"]]

    if "query_text" not in subset.columns and "query" in subset.columns:
        subset = subset.copy()
        subset["query_text"] = subset["query"]
    if "target_label" not in subset.columns:
        label_map = {}
        if "taxonomy_maps" in globals() and taxonomy in taxonomy_maps:
            label_map = taxonomy_maps[taxonomy].get("label_map", {})
        subset = subset.copy()
        subset["target_label"] = subset["target_code"].map(lambda c: label_map.get(str(c).strip(), ""))
    cols = [
        "query_text",
        "target_code",
        "target_label",
        "predicted_code",
        "predicted_label",
        "predicted_level",
        "target_level",
        "stopping_reason",
        "validation_status",
        "ambiguous",
    ]
    cols = [c for c in cols if c in subset.columns]
    return subset[cols].head(n)

sample_cases(all_eval_df, "ISCO", "classifai_validation_data", "wrong_branch", n=5)


In [ ]:
# Wrong-branch divergence report
if "all_eval_df" not in globals():
    all_eval_df = pd.concat(all_eval, ignore_index=True) if all_eval else pd.DataFrame()

wrong_branch_report = build_wrong_branch_report(all_eval_df, taxonomy_maps)
wrong_branch_report.head(20)


In [ ]:
sample_cases(all_eval_df, "ISCO", "classifai_validation_data", "under_spec", n=5)


In [ ]:
sample_cases(all_eval_df, "ISIC", "classifai_validation_data", "under_spec", n=5)

In [ ]:
# Where wrong-branch happens (root vs mid-tree + level)
if not wrong_branch_report.empty:
    parent_summary = (
        wrong_branch_report.groupby(["taxonomy", "dataset", "divergence_parent_level"])
        .size()
        .reset_index(name="wrong_branch_count")
    )
    totals = parent_summary.groupby(["taxonomy", "dataset"])["wrong_branch_count"].transform("sum")
    parent_summary["wrong_branch_share"] = parent_summary["wrong_branch_count"] / totals
    parent_summary

    level_summary = (
        wrong_branch_report.groupby(["taxonomy", "dataset", "divergence_level"])
        .size()
        .reset_index(name="wrong_branch_count")
    )
    totals = level_summary.groupby(["taxonomy", "dataset"])["wrong_branch_count"].transform("sum")
    level_summary["wrong_branch_share"] = level_summary["wrong_branch_count"] / totals
    level_summary
else:
    print("No wrong-branch rows to summarize.")
